In [1]:
import numpy as np
import pandas as pd
from collections import Counter

from sklearn.datasets import fetch_openml
from sklearn.preprocessing import StandardScaler
from common.preprocessing_util import PreprocessingUtil as pp
from sklearn.model_selection import train_test_split, KFold

In [3]:
# ==========================================
# 1. CUSTOM KNN IMPLEMENTATION
# ==========================================
class CustomKNN:
    def __init__(self, k=3):
        self.k = k

    def fit(self, X, y):
        """Memorizes the training data."""
        self.X_train = np.array(X)
        self.y_train = np.array(y)

    def predict(self, X):
        """Predicts the class for eac   h input instance."""
        X = np.array(X)
        predictions = []
        for x in X:
            # Calculate Euclidean distance using NumPy broadcasting
            distances = np.sqrt(np.sum((self.X_train - x) ** 2, axis=1))

            # Get indices of the top k closest training samples
            k_indices = np.argsort(distances)[:self.k]

            # Retrieve their corresponding labels
            k_nearest_labels = self.y_train[k_indices]

            # Determine the majority class
            most_common = Counter(k_nearest_labels).most_common(1)
            predictions.append(most_common[0][0])

        return np.array(predictions)

In [14]:
# ==========================================
# 2. CROSS-VALIDATION FOR PARAMETER TUNING
# ==========================================

def cross_validate_knn(X, y, k_values, n_splits=5, scoring='accuracy'):
    """
    Performs K-Fold cross-validation over a list of k values.

    Args:
        X         : Feature matrix (numpy array).
        y         : Labels (numpy array).
        k_values  : List of k values to evaluate.
        n_splits  : Number of folds (default: 5).
        scoring   : Metric to optimize — 'accuracy', 'f1', 'precision', or 'recall'.

    Returns:
        cv_results : Dict mapping each k to its per-fold scores and mean/std.
        best_k     : The k value with the highest mean CV score.
    """
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    cv_results = {}

    def scoring_fn(yt, yp):
        if scoring == 'accuracy':
            return Metrics.accuracy(yt, yp)
        elif scoring == 'f1':
            return Metrics.evaluate_all(yt, yp)[2]
        elif scoring == 'precision':
            return Metrics.evaluate_all(yt, yp)[0]
        elif scoring == 'recall':
            return Metrics.evaluate_all(yt, yp)[1]

    print(f"\n--- Cross-Validation ({n_splits}-Fold, metric='{scoring}') ---")
    print(f"{'k':<6} {'Fold Scores':<50} {'Mean':>8} {'Std':>8}")
    print("-" * 75)

    best_k = -1
    best_mean = -1
    for k in k_values:
        fold_scores = []

        for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X)):
            X_fold_train, X_fold_val = X[train_idx], X[val_idx]
            y_fold_train, y_fold_val = y[train_idx], y[val_idx]

            fold_scaler = StandardScaler()
            X_fold_train = fold_scaler.fit_transform(X_fold_train)
            X_fold_val   = fold_scaler.transform(X_fold_val)

            model = CustomKNN(k=k)
            model.fit(X_fold_train, y_fold_train)
            y_val_pred = model.predict(X_fold_val)

            score = scoring_fn(y_fold_val, y_val_pred)
            fold_scores.append(score)
            print(f"for k = {k} score = {score}")

        mean_score = np.mean(fold_scores)
        std_score  = np.std(fold_scores)

        if mean_score > best_mean:
            best_k = k
            best_mean = mean_score

        cv_results[k] = {'fold_scores': fold_scores, 'mean': mean_score, 'std': std_score}

        scores_str = '  '.join([f'{s:.4f}' for s in fold_scores])
        print(f"k={k:<4} [{scores_str}]  {mean_score:.4f}   ±{std_score:.4f}")

    print(f"\n✓ Best k={best_k}  (mean {scoring}={cv_results[best_k]['mean']:.4f})")
    return cv_results, best_k

In [5]:
# ==========================================
# 3. DATA PROCESSING & PIPELINE
# ==========================================

print("Loading MNIST dataset...")
mnist = fetch_openml('mnist_784', version=1, parser='auto')
X, y = mnist.data.to_numpy(), mnist.target.to_numpy().astype(int)

# Train / Test Split — scaler is NOT applied here; CV handles it fold-by-fold
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

Loading MNIST dataset...


In [6]:
# --- Feature Extraction Pipeline ---
FEATURE_STRATEGY = 'hog'

def extract_features(data, strategy='flatten'):
    if strategy == 'flatten':
        return data
    elif strategy == 'hog':
        return pp.hog_feature_extractor(pd.DataFrame(data))

In [7]:
print(f"Extracting features using {FEATURE_STRATEGY.upper()}...")
X_train_feat = extract_features(X_train, strategy=FEATURE_STRATEGY)
X_test_feat  = extract_features(X_test,  strategy=FEATURE_STRATEGY)

X_train_feat = np.array(X_train_feat)
X_test_feat  = np.array(X_test_feat)

Extracting features using HOG...
Extracting HOG features...
Extraction complete. New shape: (56000, 324)
Extracting HOG features...
Extraction complete. New shape: (14000, 324)


In [16]:
# ==========================================
# 4. CROSS-VALIDATION: FIND BEST K
# ==========================================

# Use 20% of training data for CV — enough to reliably rank k values
X_cv, _, y_cv, _ = train_test_split(
    X_train_feat, y_train,
    train_size=0.2,
    random_state=42,
    stratify=y_train
)

K_CANDIDATES = [1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21]
CV_METRIC     = 'f1' # 'accuracy', 'f1', 'precision', 'recall'
N_FOLDS       = 5

cv_results, best_k = cross_validate_knn(
    X_cv, y_cv,
    k_values=K_CANDIDATES,
    n_splits=N_FOLDS,
    scoring=CV_METRIC,
)


--- Cross-Validation (5-Fold, metric='f1') ---
k      Fold Scores                                            Mean      Std
---------------------------------------------------------------------------
for k = 1 score = 0.9330947572487149
for k = 1 score = 0.9329478558732678
for k = 1 score = 0.9345161245912901
for k = 1 score = 0.9331885196265942
for k = 1 score = 0.9379253000124298
k=1    [0.9331  0.9329  0.9345  0.9332  0.9379]  0.9343   ±0.0019
for k = 3 score = 0.941974418891903
for k = 3 score = 0.942214920160332
for k = 3 score = 0.9422353180428337
for k = 3 score = 0.9425226211229668
for k = 3 score = 0.9460680664075132
k=3    [0.9420  0.9422  0.9422  0.9425  0.9461]  0.9430   ±0.0015
for k = 5 score = 0.9492026550929685
for k = 5 score = 0.9407931325056613
for k = 5 score = 0.9452463487174703
for k = 5 score = 0.945807877591337
for k = 5 score = 0.9457095249381039
k=5    [0.9492  0.9408  0.9452  0.9458  0.9457]  0.9454   ±0.0027
for k = 7 score = 0.943551110044974
for k = 7 scor

In [17]:
# ==========================================
# 5. FINAL TRAINING WITH BEST K
# ==========================================

# Scale using the full training set now that k is fixed
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_feat)
X_test_scaled  = scaler.transform(X_test_feat)

In [18]:
print(f"\nTraining final model with best k={best_k} on full training set...")
final_model = CustomKNN(k=best_k)
final_model.fit(X_train_scaled, y_train)


Training final model with best k=5 on full training set...


In [19]:
print("Running predictions on the test set...")
y_pred = final_model.predict(X_test_scaled)

Running predictions on the test set...


In [24]:
# ==========================================
# 6. EVALUATION
# ==========================================

from common.metrices import Metrics

print("\n--- Final Model Evaluation ---")
accuracy = Metrics.accuracy(y_test, y_pred)
precision, recall, F1 = Metrics.evaluate_all(y_test, y_pred)
conf_matrix = Metrics.confusion_matrix(y_test, y_pred)

print(f"\nAccuracy:  {accuracy:.4f}")
print(f"Precision:  {precision:.4f}")
print(f"Recall:  {recall:.4f}")
print(f"F1-Score:  {F1:.4f}")

print("\nConfusion Matrix:")
print(conf_matrix)


--- Final Model Evaluation ---

Accuracy:  0.9633
Precision:  0.9644
Recall:  0.9628
F1-Score:  0.9631

Confusion Matrix:
[[1372    3    0    2    0    0    3    0    0    1]
 [   1 1548    7    4    4    0    1    9    1    0]
 [   4    0 1321   48    2    0    0    7   13    3]
 [   4    1    6 1384    0    6    2    4   14    7]
 [   2    1    1    0 1266    0    8   11    1   75]
 [   3    0    1   27    2 1180   19    0   25    6]
 [   3    0    1    0    2    1 1364    0    4    0]
 [   2    3    4    5    5    0    0 1392    2   46]
 [  15    6    2    5    1    3   11    3 1302   17]
 [   2    2    0    9    1    0    1   17    2 1357]]


In [28]:
# ==========================================
# 7. Load CNN data
# ==========================================

X_cnn = np.load("../../common/CNN_data/X.npy")
y_cnn = np.load("../../common/CNN_data/Y.npy")

# Train / Test Split
X_train_cnn, X_test_cnn, y_train_cnn, y_test_cnn = train_test_split(
    X_cnn, y_cnn, test_size=0.2, random_state=42, stratify=y
)

In [29]:
# ==========================================
# 8. Train the model
# ==========================================

print(f"\nTraining model with k={best_k} on CNN dataset...")
final_model = CustomKNN(k=best_k)
final_model.fit(X_train_cnn, y_train_cnn)


Training model with k=5 on CNN dataset...


In [30]:
# ==========================================
# 9. Test the model
# ==========================================

print("Running predictions on the test set...")
y_pred_cnn = final_model.predict(X_test_cnn)

Running predictions on the test set...


In [31]:
# ==========================================
# 10. EVALUATION
# ==========================================

from common.metrices import Metrics

print("\n--- Final Model Evaluation ---")
accuracy = Metrics.accuracy(y_test_cnn, y_pred_cnn)
precision, recall, F1 = Metrics.evaluate_all(y_test_cnn, y_pred_cnn)
conf_matrix = Metrics.confusion_matrix(y_test_cnn, y_pred_cnn)

print(f"\nAccuracy:  {accuracy:.4f}")
print(f"Precision:  {precision:.4f}")
print(f"Recall:  {recall:.4f}")
print(f"F1-Score:  {F1:.4f}")

print("\nConfusion Matrix:")
print(conf_matrix)


--- Final Model Evaluation ---

Accuracy:  0.9656
Precision:  0.9656
Recall:  0.9651
F1-Score:  0.9652

Confusion Matrix:
[[1380    0    0    3    0    0    8    0    0    4]
 [   0 1505    2    0    6    0    0    7    0    0]
 [   6    4 1289    7   10   17   25   28    5    3]
 [   0    0   15 1403    2   18    1    9    2    2]
 [   0    5    0    1 1356    1    3    5    0    4]
 [   2    2    8   41    3 1199   12    8    0    1]
 [  13    3    9    3    2    2 1340    2    1    1]
 [   0    5    4    0   13    0    0 1408    0    3]
 [   6    4    7    6    6    9   11    2 1292   21]
 [  10    4   11    8   12    5    5   12    2 1346]]
